In [ ]:
# config bootstrap (auto-added): resolve repo paths from config.py
import os as _os, sys as _sys
_h = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_h, 'config.py')) and _os.path.dirname(_h) != _h:
    _h = _os.path.dirname(_h)
_sys.path.insert(0, _h)
import config as _cfg

# Fusion Classifier — Combining All Three Modules

Combines scores from:
1. **BLIP ITM** — image-text matching probability
2. **DeBERTa NLI** — article→caption entailment probability
3. **SightEngine** — AI-generated image detection score

Trains a Logistic Regression on 1000 samples with 5-fold cross-validation.

In [1]:
import json
import os
import base64
import time

import torch
import requests
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import (
    BlipProcessor, BlipForImageTextRetrieval,
    AutoTokenizer, AutoModelForSequenceClassification,
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
)
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

d:\Pics Can Lie\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.6.0+cu124
CUDA: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
# ── Config ──
DATASET_ROOT     = _os.path.join(str(_cfg.ROOT), 'datasets', 'dataset')
ANNOTATIONS_PATH = os.path.join(DATASET_ROOT, "data", "NewsClipPings", "merged_balanced", "train.json")
METADATA_PATH    = os.path.join(DATASET_ROOT, "data", "NewsClipPings", "metadata", "train.json")
IMAGE_BASE       = os.path.join(DATASET_ROOT, "origin")
ARTICLE_BASE     = os.path.join(DATASET_ROOT, "origin")

NUM_PER_CLASS = 1000  # 1000 real + 1000 fake = 2000 total

def resolve_image_path(meta_image_path: str) -> str:
    rel = meta_image_path.replace("visual_news/", "", 1)
    return os.path.join(IMAGE_BASE, rel)

def resolve_article_path(meta_article_path: str) -> str:
    rel = meta_article_path.replace("visual_news/", "", 1)
    return os.path.join(ARTICLE_BASE, rel)

print("Config ready.")

Config ready.


In [3]:
# ── Load 200 samples (100 real + 100 fake) ──
with open(ANNOTATIONS_PATH, "r", encoding="utf-8") as f:
    annotations = json.load(f)["annotations"]
with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

real_samples, fake_samples = [], []

for ann in annotations:
    if len(real_samples) >= NUM_PER_CLASS and len(fake_samples) >= NUM_PER_CLASS:
        break

    art_id = str(ann["id"])
    img_id = str(ann["image_id"])
    if art_id not in metadata or img_id not in metadata:
        continue

    img_path = resolve_image_path(metadata[img_id]["image_path"])
    art_path = resolve_article_path(metadata[img_id]["article_path"])
    if not os.path.isfile(img_path) or not os.path.isfile(art_path):
        continue

    with open(art_path, "r", encoding="utf-8", errors="replace") as f:
        article_text = f.read(2000)
    if len(article_text.strip()) < 50:
        continue

    entry = {
        "article_id": ann["id"],
        "image_id": ann["image_id"],
        "caption": metadata[art_id]["caption"],
        "image_path": img_path,
        "article_text": article_text,
        "falsified": ann["falsified"],
    }

    if not ann["falsified"] and len(real_samples) < NUM_PER_CLASS:
        real_samples.append(entry)
    elif ann["falsified"] and len(fake_samples) < NUM_PER_CLASS:
        fake_samples.append(entry)

samples = real_samples + fake_samples
print(f"Loaded {len(real_samples)} real + {len(fake_samples)} fake = {len(samples)} samples")

Loaded 1000 real + 1000 fake = 2000 samples


In [4]:
# ── Module 1: BLIP ITM scores ──
print("Loading BLIP ITM model...")
blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-itm-base-coco")
blip_model = BlipForImageTextRetrieval.from_pretrained("Salesforce/blip-itm-base-coco")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
blip_model = blip_model.to(device).eval()
print(f"BLIP loaded on {device}")

itm_scores = []
for s in tqdm(samples, desc="BLIP ITM"):
    image = Image.open(s["image_path"]).convert("RGB")
    inputs = blip_processor(images=image, text=s["caption"], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = blip_model(**inputs, use_itm_head=True)
        probs = torch.softmax(outputs.itm_score, dim=1)
        itm_scores.append(probs[0, 1].item())

# Free GPU memory
del blip_model, blip_processor
torch.cuda.empty_cache()
print(f"BLIP done. Scores: min={min(itm_scores):.4f}, max={max(itm_scores):.4f}, mean={np.mean(itm_scores):.4f}")

Loading BLIP ITM model...


The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 472/472 [00:00<00:00, 36095.16it/s]
BlipForImageTextRetrieval LOAD REPORT from: Salesforce/blip-itm-base-coco
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_encoder.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BLIP loaded on cuda


BLIP ITM: 100%|██████████| 2000/2000 [02:05<00:00, 15.90it/s]

BLIP done. Scores: min=0.0000, max=1.0000, mean=0.4363


In [5]:
# ── Module 2: DeBERTa NLI entailment scores ──
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re

def extract_top_sentences(article: str, caption: str, top_k: int = 3) -> str:
    """Extract the top-k most relevant sentences from the article using TF-IDF."""
    sentences = [s.strip() for s in re.split(r'[.!?]+', article) if len(s.strip()) > 20]
    if len(sentences) <= top_k:
        return article
    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf = vectorizer.fit_transform([caption] + sentences)
    sims = cosine_similarity(tfidf[0:1], tfidf[1:])[0]
    top_idx = sims.argsort()[-top_k:][::-1]
    return ". ".join(sentences[i] for i in sorted(top_idx)) + "."

print("Loading DeBERTa NLI model...")
nli_tokenizer = AutoTokenizer.from_pretrained("cross-encoder/nli-deberta-v3-large")
nli_model = AutoModelForSequenceClassification.from_pretrained("cross-encoder/nli-deberta-v3-large").to(device)
nli_model.eval()
print(f"DeBERTa loaded on {device}")

entailment_scores = []
for s in tqdm(samples, desc="DeBERTa NLI"):
    premise = extract_top_sentences(s["article_text"], s["caption"])
    inputs = nli_tokenizer(
        premise, s["caption"],
        return_tensors="pt", truncation=True, max_length=512, padding=True
    ).to(device)
    with torch.no_grad():
        logits = nli_model(**inputs).logits
        probs = torch.softmax(logits, dim=1)[0]
        entailment_scores.append(probs[1].item())  # index 1 = entailment

del nli_model, nli_tokenizer
torch.cuda.empty_cache()
print(f"NLI done. Scores: min={min(entailment_scores):.4f}, max={max(entailment_scores):.4f}, mean={np.mean(entailment_scores):.4f}")

Loading DeBERTa NLI model...


Loading weights: 100%|██████████| 394/394 [00:00<00:00, 3849.86it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DeBERTa loaded on cuda


DeBERTa NLI: 100%|██████████| 2000/2000 [04:06<00:00,  8.11it/s]


NLI done. Scores: min=0.0000, max=0.9992, mean=0.1837


In [6]:
# ── Module 3: SightEngine AI-generated scores (with disk caching) ──
SIGHTENGINE_URL   = "https://api.sightengine.com/1.0/check.json"
SIGHTENGINE_USER  = "1409625234"
SIGHTENGINE_SECRET = "WFwczdygGNWodnQ2Mn565QqXfumU866Q"
SE_CACHE_PATH     = _os.path.join(str(_cfg.ROOT), 'cache', 'sightengine_cache.json')

# Load cache
if os.path.isfile(SE_CACHE_PATH):
    with open(SE_CACHE_PATH, "r") as f:
        se_cache = json.load(f)
    print(f"Loaded {len(se_cache)} cached SightEngine results")
else:
    se_cache = {}

def sightengine_cached(image_path: str) -> float:
    """Get SightEngine AI score with disk caching."""
    cache_key = os.path.basename(image_path)
    if cache_key in se_cache:
        return se_cache[cache_key]

    params = {
        "models": "genai",
        "api_user": SIGHTENGINE_USER,
        "api_secret": SIGHTENGINE_SECRET,
    }
    with open(image_path, "rb") as img_file:
        files = {"media": (os.path.basename(image_path), img_file)}
        response = requests.post(SIGHTENGINE_URL, files=files, data=params, timeout=30)

    response.raise_for_status()
    data = response.json()

    if data.get("status") != "success":
        raise RuntimeError(f"API error: {data}")

    ai_score = data.get("type", {}).get("ai_generated", 0.0)

    se_cache[cache_key] = ai_score
    with open(SE_CACHE_PATH, "w") as f:
        json.dump(se_cache, f)

    return ai_score

# Run on all 1000 samples
sightengine_scores = []
errors = 0
for i, s in enumerate(tqdm(samples, desc="SightEngine")):
    try:
        score = sightengine_cached(s["image_path"])
        sightengine_scores.append(score)
    except Exception as e:
        errors += 1
        sightengine_scores.append(0.0)
        if "429" in str(e) or "rate" in str(e).lower() or "limit" in str(e).lower():
            print(f"\nRate limited at sample {i+1}. {len(se_cache)} cached so far.")
            print("Re-run this cell later to continue from cache.")
            break
    time.sleep(0.3)

# Pad if rate-limited
while len(sightengine_scores) < len(samples):
    sightengine_scores.append(0.0)

print(f"\nSightEngine done. Cached: {len(se_cache)}, Errors: {errors}")
print(f"Scores: min={min(sightengine_scores):.6f}, max={max(sightengine_scores):.6f}, mean={np.mean(sightengine_scores):.6f}")

Loaded 11 cached SightEngine results


SightEngine:   0%|          | 0/300 [00:00<?, ?it/s]

SightEngine:  49%|████▊     | 146/300 [03:03<03:14,  1.26s/it]


Rate limited at sample 147. 132 cached so far.
Re-run this cell later to continue from cache.

SightEngine done. Cached: 132, Errors: 2
Scores: min=0.000000, max=0.900000, mean=0.008683


In [6]:
# ── Build feature matrix (BLIP ITM + DeBERTa only) ──
labels = [1 if s["falsified"] else 0 for s in samples]

df = pd.DataFrame({
    "article_id": [s["article_id"] for s in samples],
    "image_id": [s["image_id"] for s in samples],
    "itm_score": itm_scores,
    "entailment_score": entailment_scores,
    "label": labels,
    "label_str": ["FAKE" if l else "REAL" for l in labels],
})

print("Feature summary:")
print(df.groupby("label_str")[["itm_score", "entailment_score"]].agg(["mean", "std"]).round(4))
print(f"\nTotal samples: {len(df)} ({(df['label']==0).sum()} real, {(df['label']==1).sum()} fake)")

df.to_csv(_os.path.join(str(_cfg.ROOT), 'features', 'fusion_features_2000.csv'), index=False)
print("Saved to fusion_features_2000.csv")

Feature summary:
          itm_score         entailment_score        
               mean     std             mean     std
label_str                                           
FAKE         0.2203  0.3472           0.0875  0.2214
REAL         0.6523  0.3937           0.2798  0.4257

Total samples: 2000 (1000 real, 1000 fake)
Saved to fusion_features_2000.csv


In [7]:
# ── 5-Fold Cross-Validation (BLIP ITM + DeBERTa) ──
from sklearn.model_selection import StratifiedKFold

X = df[["itm_score", "entailment_score"]].values
y = df["label"].values

print(f"Features: BLIP ITM + DeBERTa Entailment (2 features)")
print(f"Samples: {len(X)} ({sum(y==0)} real, {sum(y==1)} fake)")
print(f"Method: 5-fold stratified cross-validation\n")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_results = []
all_y_true = []
all_y_pred = []
all_y_prob = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    clf = LogisticRegression(random_state=42, max_iter=1000)
    clf.fit(X_train_scaled, y_train)

    y_pred = clf.predict(X_test_scaled)
    y_prob = clf.predict_proba(X_test_scaled)[:, 1]

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)

    fold_results.append({"fold": fold+1, "accuracy": acc, "precision": prec, "recall": rec, "f1": f1})
    all_y_true.extend(y_test)
    all_y_pred.extend(y_pred)
    all_y_prob.extend(y_prob)

    print(f"  Fold {fold+1}: Acc={acc:.1%}  Prec={prec:.1%}  Rec={rec:.1%}  F1={f1:.1%}  "
          f"(train={len(train_idx)}, test={len(test_idx)})")

fold_df = pd.DataFrame(fold_results)
print(f"\n{'=' * 65}")
print(f"5-FOLD CV RESULTS (BLIP ITM + DeBERTa NLI) — {len(X)} samples")
print(f"{'=' * 65}")
print(f"  Mean Accuracy  : {fold_df['accuracy'].mean():.2%} (+/- {fold_df['accuracy'].std():.2%})")
print(f"  Mean Precision : {fold_df['precision'].mean():.2%} (+/- {fold_df['precision'].std():.2%})")
print(f"  Mean Recall    : {fold_df['recall'].mean():.2%} (+/- {fold_df['recall'].std():.2%})")
print(f"  Mean F1 Score  : {fold_df['f1'].mean():.2%} (+/- {fold_df['f1'].std():.2%})")
print(f"{'=' * 65}")

all_y_true = np.array(all_y_true)
all_y_pred = np.array(all_y_pred)
all_y_prob = np.array(all_y_prob)

print(f"\nAggregated classification report (all {len(all_y_true)} predictions):")
print(classification_report(all_y_true, all_y_pred, target_names=["REAL", "FAKE"]))

cm = confusion_matrix(all_y_true, all_y_pred)
cm_df = pd.DataFrame(cm, index=["Actual REAL", "Actual FAKE"], columns=["Pred REAL", "Pred FAKE"])
display(cm_df)

# Feature weights from last fold
print(f"\nFeature weights (last fold):")
for name, coef in zip(["BLIP ITM", "DeBERTa Entailment"], clf.coef_[0]):
    print(f"  {name:>20s}: {coef:+.4f}")

Features: BLIP ITM + DeBERTa Entailment (2 features)
Samples: 2000 (1000 real, 1000 fake)
Method: 5-fold stratified cross-validation

  Fold 1: Acc=76.8%  Prec=77.7%  Rec=75.0%  F1=76.3%  (train=1600, test=400)
  Fold 2: Acc=75.0%  Prec=77.2%  Rec=71.0%  F1=74.0%  (train=1600, test=400)
  Fold 3: Acc=77.0%  Prec=77.0%  Rec=77.0%  F1=77.0%  (train=1600, test=400)
  Fold 4: Acc=75.0%  Prec=73.4%  Rec=78.5%  F1=75.8%  (train=1600, test=400)
  Fold 5: Acc=76.2%  Prec=74.9%  Rec=79.0%  F1=76.9%  (train=1600, test=400)

5-FOLD CV RESULTS (BLIP ITM + DeBERTa NLI) — 2000 samples
  Mean Accuracy  : 76.00% (+/- 0.95%)
  Mean Precision : 76.03% (+/- 1.84%)
  Mean Recall    : 76.10% (+/- 3.25%)
  Mean F1 Score  : 76.01% (+/- 1.23%)

Aggregated classification report (all 2000 predictions):
              precision    recall  f1-score   support

        REAL       0.76      0.76      0.76      1000
        FAKE       0.76      0.76      0.76      1000

    accuracy                           0.76     

,Pred REAL,Pred FAKE
Actual REAL,759,241
Actual FAKE,239,761



Feature weights (last fold):
              BLIP ITM: -1.1579
    DeBERTa Entailment: -0.6939


In [ ]:
# ── Visualizations ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Confusion matrix heatmap (aggregated)
ax = axes[0]
im = ax.imshow(cm, cmap="Blues", aspect="auto")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Pred REAL", "Pred FAKE"])
ax.set_yticklabels(["Actual REAL", "Actual FAKE"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=18, fontweight="bold")
ax.set_title("Confusion Matrix (All Folds)")

# 2. Per-fold metrics bar chart
ax = axes[1]
x = np.arange(5)
w = 0.2
ax.bar(x - 1.5*w, fold_df["accuracy"], w, label="Accuracy", color="#3498db", edgecolor="white")
ax.bar(x - 0.5*w, fold_df["precision"], w, label="Precision", color="#2ecc71", edgecolor="white")
ax.bar(x + 0.5*w, fold_df["recall"], w, label="Recall", color="#e67e22", edgecolor="white")
ax.bar(x + 1.5*w, fold_df["f1"], w, label="F1", color="#e74c3c", edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels([f"Fold {i+1}" for i in range(5)])
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("Per-Fold Metrics")
ax.legend(fontsize=8)

# 3. Fusion probability distribution (aggregated)
ax = axes[2]
real_probs = all_y_prob[all_y_true == 0]
fake_probs = all_y_prob[all_y_true == 1]
ax.hist(real_probs, bins=15, alpha=0.7, color="#2ecc71", label="REAL", edgecolor="white")
ax.hist(fake_probs, bins=15, alpha=0.7, color="#e74c3c", label="FAKE", edgecolor="white")
ax.axvline(x=0.5, color="orange", linestyle="--", linewidth=2, label="Threshold")
ax.set_xlabel("Predicted FAKE Probability")
ax.set_ylabel("Count")
ax.set_title("Classifier Output (All Folds)")
ax.legend()

mean_f1 = fold_df["f1"].mean()
mean_acc = fold_df["accuracy"].mean()
plt.suptitle(f"Fusion Classifier (5-Fold CV) — Mean Acc: {mean_acc:.1%} | Mean F1: {mean_f1:.1%}",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("fusion_classifier_5fold.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Compare: individual modules vs fusion ──
print(f"{'Module':<30s} {'Accuracy':>10s} {'Precision':>10s} {'Recall':>10s} {'F1':>10s}")
print("-" * 72)

# BLIP ITM alone
blip_pred = (X[:, 0] < 0.5).astype(int)
print(f"{'BLIP ITM (threshold=0.5)':<30s} "
      f"{accuracy_score(y, blip_pred):>10.1%} "
      f"{precision_score(y, blip_pred):>10.1%} "
      f"{recall_score(y, blip_pred):>10.1%} "
      f"{f1_score(y, blip_pred):>10.1%}")

# DeBERTa alone
nli_pred = (X[:, 1] < 0.5).astype(int)
print(f"{'DeBERTa NLI (threshold=0.5)':<30s} "
      f"{accuracy_score(y, nli_pred):>10.1%} "
      f"{precision_score(y, nli_pred):>10.1%} "
      f"{recall_score(y, nli_pred):>10.1%} "
      f"{f1_score(y, nli_pred):>10.1%}")

# Fusion
print(f"{'FUSION 5-Fold CV (LogReg)':<30s} "
      f"{fold_df['accuracy'].mean():>10.1%} "
      f"{fold_df['precision'].mean():>10.1%} "
      f"{fold_df['recall'].mean():>10.1%} "
      f"{fold_df['f1'].mean():>10.1%}")
print("-" * 72)